In [ ]:
import os
import cv2
import pandas as pd

# ============================================================
# Select Dataset
# ============================================================

dataset = input("Which dataset do you want to label? (train / valid / test): ").strip().lower()

BASE_FOLDER = r"data_set_generator_for_cnns_only"

if dataset == "train":
    IMAGE_FOLDER = os.path.join(BASE_FOLDER, "train")
    CSV_FILE = os.path.join(BASE_FOLDER, "train.csv")

elif dataset in ["valid", "validation", "val"]:
    IMAGE_FOLDER = os.path.join(BASE_FOLDER, "valid")
    CSV_FILE = os.path.join(BASE_FOLDER, "valid.csv")


elif dataset == "test":
    IMAGE_FOLDER = os.path.join(BASE_FOLDER, "test")
    CSV_FILE = os.path.join(BASE_FOLDER, "test.csv")

else:
    print("Invalid dataset.")
    exit()

SUPPORTED_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")

# ============================================================
# Resume Support
# ============================================================

if os.path.exists(CSV_FILE):
    df = pd.read_csv(CSV_FILE, dtype={"label": str})
    labeled_images = set(df["image"])
else:
    df = pd.DataFrame(columns=["image", "label"])
    labeled_images = set()

# ============================================================
# Image List
# ============================================================

images = sorted([
    img for img in os.listdir(IMAGE_FOLDER)
    if img.lower().endswith(SUPPORTED_EXTENSIONS)
])

print(f"\nDataset : {dataset}")
print(f"Total Images : {len(images)}")
print(f"Already Labeled : {len(labeled_images)}")

# ============================================================
# Labeling Loop
# ============================================================

for index, image_name in enumerate(images):

    if image_name in labeled_images:
        continue

    image_path = os.path.join(IMAGE_FOLDER, image_name)

    img = cv2.imread(image_path)

    if img is None:
        print(f"Could not read {image_name}")
        continue

    # Resize only for display
    h, w = img.shape[:2]
    scale = min(900 / w, 400 / h)
    display = cv2.resize(img, (int(w * scale), int(h * scale)))

    cv2.imshow("Meter Labeling Tool", display)
    cv2.waitKey(1)

    print(f"\n[{index+1}/{len(images)}] {image_name}")

    while True:

        label = input("Enter 5-digit reading (q = quit, s = skip): ").strip()

        if label.lower() == "q":
            cv2.destroyAllWindows()
            df.to_csv(CSV_FILE, index=False)
            print("\nProgress Saved.")
            exit()

        if label.lower() == "s":
            break

        if len(label) == 5 and label.isdigit():
            df.loc[len(df)] = [image_name, label]
            df.to_csv(CSV_FILE, index=False)
            break

        print("Invalid label! Please enter exactly 5 digits.")

cv2.destroyAllWindows()

print("\n===================================")
print("All images have been labeled!")
print("CSV saved to:")
print(CSV_FILE)
print("===================================")